In [ ]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_295K_278464_17O_opt_magres_new.magres') #latest magres file from 2025

In [27]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [28]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [29]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [30]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-58.62788709 168.40679868 -80.11241991]
 [175.61811553  49.04275794  28.95345044]
 [  9.62496898 -25.67949218 -52.76157831]]

17O2 sigma:
 [[ -58.62788709 -168.40679868   80.11241991]
 [-175.61811553   49.04275794   28.95345044]
 [  -9.62496898  -25.67949218  -52.76157831]]

17O3 sigma:
 [[-58.62788709 168.40679868  80.11241991]
 [175.61811553  49.04275794 -28.95345044]
 [ -9.62496898  25.67949218 -52.76157831]]

17O4 sigma:
 [[ -58.62788709 -168.40679868  -80.11241991]
 [-175.61811553   49.04275794  -28.95345044]
 [   9.62496898   25.67949218  -52.76157831]]

17O5 sigma:
 [[ -47.44057414  228.4474648    55.076171  ]
 [ 197.49554429  102.00730098  -88.59347837]
 [  16.93245242  -50.39796985 -180.05968488]]

17O6 sigma:
 [[ -47.44057414 -228.4474648   -55.076171  ]
 [-197.49554429  102.00730098  -88.59347837]
 [ -16.93245242  -50.39796985 -180.05968488]]

17O7 sigma:
 [[ -47.44057414  228.4474648   -55.076171  ]
 [ 197.49554429  102.00730098   88.59347837]
 [ -16.93245242

In [31]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.352899935504245

17O2 sigma:
 6.35289993550426

17O3 sigma:
 6.35289993550424

17O4 sigma:
 6.352899935504253

17O5 sigma:
 8.335934183823582

17O6 sigma:
 8.335934183823564

17O7 sigma:
 8.335934183823579

17O8 sigma:
 8.33593418382361



In [32]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

atom_label = 4
CS_total[:,:] = atoms.species('O').ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('O')[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = -0.0256 #electric quadrupole moment for O17 in barn
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[-4.222  0.94  -0.785]
 [ 0.94  -3.887  1.557]
 [-0.785  1.557  8.109]]

CS Tensor:
 [[ -47.441  228.447   55.076]
 [ 197.496  102.007  -88.593]
 [  16.932  -50.398 -180.06 ]]

CS isotropic Tensor:
 [[-41.831   0.      0.   ]
 [  0.    -41.831   0.   ]
 [  0.      0.    -41.831]]

CS symmetric Tensor:
 [[ -47.441  212.972   36.004]
 [ 212.972  102.007  -69.496]
 [  36.004  -69.496 -180.06 ]]

CS antisymmetric Tensor:
 [[  0.     15.476  19.072]
 [-15.476   0.    -19.098]
 [-19.072  19.098   0.   ]]


In [33]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [-5.20535768 -3.13709119  8.34244887] 

 Unsorted Eigenvectors:
 [[-0.73470896  0.67632344 -0.05281424]
 [ 0.66743388  0.73458807  0.12211629]
 [-0.12138682 -0.05446992  0.99110961]] 

Sorted Eigenvalues: 
 [-3.13709119 -5.20535768  8.34244887] 

Sorted Eigenvectors: 
 [[ 0.67632344 -0.73470896 -0.05281424]
 [ 0.73458807  0.66743388  0.12211629]
 [-0.05446992 -0.12138682  0.99110961]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 256.00835187 -120.99629148 -260.50501843] 

 Unsorted Eigenvectors:
 [[-0.56574143 -0.58502488 -0.58110457]
 [-0.82029116  0.32748756  0.46890757]
 [ 0.08401808 -0.74195539  0.66516402]] 

Sorted Eigenvalues: 
 [-120.99629148 -260.50501843  256.00835187] 

Sorted Eigenvectors: 
 [[-0.58502488 -0.58110457 -0.56574143]
 [ 0.32748756  0.46890757 -0.82029116]
 [-0.74195539  0.66516402  0.08401808]] 



In [34]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -3.1370911901930727 -5.205357675778691 8.342448865971747
CSA Tensor Components δyy, δxx, δzz: 
 -120.99629147528037 -260.5050184343854 256.0083518684129


In [35]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        8.34245
etaq            0.247921
iso_cs (ppm)  -41.831
csa (ppm)     297.839
etas            0.468403


In [36]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.73470896  0.67632344 -0.05281424]
 [ 0.66743388  0.73458807  0.12211629]
 [-0.12138682 -0.05446992  0.99110961]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
24.167209282117927 7.645752097191747 66.61193487505577 

Direction cosine csa: 

[[-0.58110457 -0.58502488 -0.56574143]
 [ 0.46890757  0.32748756 -0.82029116]
 [ 0.66516402 -0.74195539  0.08401808]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-48.12373010038013 85.18043720780439 -55.40660420582955 



In [37]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -41.64596272028763 chi: 89.2563135738765 xi: -81.83377412176924 



**Rotation of tensors Crystal--> Tenon Frame**

In [38]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[ -47.44057414  212.97150455   36.00431171]
 [ 212.97150455  102.00730098  -69.49572411]
 [  36.00431171  -69.49572411 -180.05968488]]
CSA Tensor in Tenon Frame: 
 [[ 162.72897542 -144.63523547 -111.4259363 ]
 [-144.63523547 -210.47611566   47.68508823]
 [-111.4259363    47.68508823  -77.7458178 ]]
Quad Tensor in Crystal Frame: 
 [[-4.22151538  0.94017806 -0.78534934]
 [ 0.94017806 -3.88725033  1.55694211]
 [-0.78534934  1.55694211  8.10876571]]
Quad Tensor in Tenon Frame: 
 [[-3.83690789 -0.72162418 -0.66833977]
 [-0.72162418  2.67863207 -6.37809411]
 [-0.66833977 -6.37809411  1.15827582]]
